# Semana 13 — Defesa em profundidade para a VPC Smart N1

**Roteiro:** revisão expressa da VPC pública/privada da Semana 12; foco em reduzir exposição, restringir caminhos e administrar por identidade. A aula prioriza leitura e desenho; não altere recursos ativos sem confirmar a conta e o laboratório.

## Objetivos

- relembrar CIDR, sub-redes, rotas, IGW e NAT;
- distinguir Security Group de Network ACL;
- propor administração sem SSH aberto à internet;
- reconhecer VPC endpoints e privilégio mínimo;
- documentar fluxos permitidos e bloqueados.

## Revisão rápida da Semana 12 — 8 minutos

Desenhe navegador → frontend público → backend privado. Responda: qual rota torna a sub-rede pública? A instância privada precisa IP público para saída via NAT? SG é stateful? NAT abre entrada não solicitada?

**Gabarito:** rota para IGW; não precisa IP público para iniciar saída via NAT; SG é stateful; NAT não é regra de entrada não solicitada.

## Fronteiras de confiança

```text
Internet → SG frontend (HTTPS) → frontend público
Frontend → SG backend (porta da API, origem = SG frontend)
Backend → banco/serviços necessários (sem exposição pública)
Administração → Session Manager, se disponível (sem SSH público)
```

“Privado” não significa seguro por si só: revise rota, regras, IAM, credenciais, atualizações e logs.

## Security Group e Network ACL

| Controle | Associado a | Stateful? | Uso típico |
|---|---|---|---|
| Security Group | interface/recurso | sim | permitir portas e origens necessárias |
| Network ACL | sub-rede | não | barreira adicional de entrada e saída |

Como NACL é stateless, o tráfego de retorno também precisa ser permitido. Regras em camadas podem bloquear fluxo que o SG permite; mantenha a política legível.

## Exemplo — regras encadeadas

| Recurso | Entrada | Origem |
|---|---|---|
| Frontend | TCP 443 | faixa pública definida para o exercício |
| Backend | porta da API, ex. TCP 5000 | SG frontend |
| Banco | porta do banco | SG backend |
| SSH | nenhuma regra pública | usar Session Manager se configurado |

Evite `0.0.0.0/0` para administração ou banco. SG pode referenciar outro SG como origem.

## Egress e administração

Revise se o backend precisa de saída ampla. Liste destinos reais: atualização, logs/monitoramento, broker e APIs AWS. Bloquear sem inventário pode quebrar operações.

Session Manager pode oferecer shell sem abrir SSH público, mas exige agente, instance profile e conectividade aos endpoints (NAT ou VPC endpoints). Se indisponível no Learner Lab, documente a limitação e restrinja eventual acesso temporário ao IP administrativo, removendo a regra ao final.

## VPC endpoints e IAM

VPC endpoints permitem acessar alguns serviços AWS sem usar caminho público. Há gateway endpoints (como S3/DynamoDB) e interface endpoints (como Systems Manager/CloudWatch Logs); disponibilidade e cobrança variam. Revise DNS, rota, SG, política do endpoint e IAM.

IAM autoriza chamadas a APIs AWS; não substitui SG, NACL nem autenticação da aplicação. Use papel de instância com ações e recursos mínimos, identidades individuais e nenhum segredo no repositório.

## Exemplo — matriz de fluxo

| Origem → destino | Decisão | Controle/teste |
|---|---|---|
| Internet → frontend:443 | permitir conforme exercício | SG + rota; abrir HTTPS |
| Internet → backend:5000 | bloquear | sem exposição/entrada; teste negativo |
| SG frontend → backend:5000 | permitir | referência de SG; health interno |
| SG frontend → banco:5432 | bloquear | frontend não acessa banco direto |
| Backend → S3 | conforme necessidade | endpoint + política IAM; objeto autorizado |

Teste negativo é evidência de segurança tão importante quanto teste positivo.

In [ ]:
# Matriz de raciocínio; este código NÃO altera a AWS.
fluxos = [
    ("internet", "frontend", 443, True),
    ("internet", "backend", 5000, False),
    ("sg-frontend", "backend", 5000, True),
    ("sg-frontend", "banco", 5432, False),
]
for origem, destino, porta, permitir in fluxos:
    decisao = "PERMITIR" if permitir else "BLOQUEAR"
    print(f"{decisao:9} {origem:12} -> {destino:8} TCP/{porta}")

## Diagnóstico em camadas

- navegador sem resposta: IP/DNS, rota pública, IGW, SG frontend, processo/listener;
- frontend não alcança API: endpoint interno, SG backend, porta/processo, logs;
- backend não alcança AWS: rota/NAT/endpoint, DNS, SG/NACL, IAM e política do endpoint;
- SSM offline: instance profile, agente, DNS/rota, endpoints e permissões.

Mude uma coisa por vez. Não corrija timeout abrindo todas as portas.

## Exercício — endurecer a Smart N1

Uma arquitetura tem backend exposto e SSH aberto para qualquer origem. Produza um diagrama de camadas; tabela SG de entrada/saída; decisão sobre NACL e retorno; método administrativo; endpoints necessários; três fluxos permitidos e três bloqueados; evidências e plano de retorno. Identifique limitações do Learner Lab antes de propor execução.

## Checklist e referências oficiais

- [ ] backend/banco sem exposição pública desnecessária;
- [ ] origem e porta mínimas nas regras;
- [ ] egress e identidade da instância revisados;
- [ ] acesso administrativo individual e registrado;
- [ ] fluxo autorizado e bloqueado testados;
- [ ] recursos do laboratório encerrados.

- AWS, Security Groups: <https://docs.aws.amazon.com/vpc/latest/userguide/vpc-security-groups.html>
- AWS, Network ACLs: <https://docs.aws.amazon.com/vpc/latest/userguide/vpc-network-acls.html>
- AWS, Session Manager: <https://docs.aws.amazon.com/systems-manager/latest/userguide/session-manager.html>
- AWS, PrivateLink: <https://docs.aws.amazon.com/vpc/latest/privatelink/what-is-privatelink.html>
- AWS, IAM best practices: <https://docs.aws.amazon.com/IAM/latest/UserGuide/best-practices.html>